# 🏏 IPL Auction Player Ranking — Sorting Algorithms for Business Prioritization

**Session 1 | Industrial Use Case 2 | DSA & ML for Business**

---

### Business Context
- IPL teams rank **500+ players** before each mega auction
- Total purse per team: **₹100 Crore (~$12M)**
- Wrong ranking = overpaying by **₹10-30 Cr** per player
- Multi-criteria problem: performance, form, injury risk, role fit

### What You'll Learn
1. Build a **weighted composite score** for multi-criteria ranking
2. Compare **Selection Sort, Insertion Sort, and QuickSort** on real player data
3. Perform **sensitivity analysis** on weight parameters
4. Apply **budget-constrained optimization** for squad selection

## Step 1: Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import copy

# Load dataset
df = pd.read_csv("../datasets/ipl_auction_players.csv")
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nRoles: {df['role'].unique()}")
df.head(10)

In [ ]:
# Quick EDA
print("=== Dataset Summary ===")
print(df.describe().round(2))
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nPlayers per role:\n{df['role'].value_counts()}")

## Step 2: Weighted Composite Score

Create a configurable multi-criteria ranking score:

**`composite_score = w1 × strike_rate + w2 × batting_avg + w3 × recent_form - w4 × injury_risk + role_bonus`**

Different team strategies produce different weight vectors:
- **Aggressive team:** high weight on strike rate → prioritize finishers
- **Balanced team:** equal weights across all metrics
- **Conservative team:** high weight on batting average + low injury → prioritize consistency

In [ ]:
# Normalize features to 0-1 range for fair weighting
from sklearn.preprocessing import MinMaxScaler

features_to_normalize = ['strike_rate', 'batting_avg', 'recent_form_score', 'injury_risk']
scaler = MinMaxScaler()
df_norm = df.copy()
df_norm[features_to_normalize] = scaler.fit_transform(df[features_to_normalize])

# Role bonus mapping
role_bonus = {
    'opener': 0.05,
    'middle_order': 0.03,
    'finisher': 0.08,
    'bowler': 0.04,
    'allrounder': 0.10
}

# Define three team strategies
strategies = {
    'Aggressive': {'w1': 0.35, 'w2': 0.15, 'w3': 0.30, 'w4': 0.20},  # High strike rate + form
    'Balanced':   {'w1': 0.25, 'w2': 0.25, 'w3': 0.25, 'w4': 0.25},  # Equal weights
    'Conservative': {'w1': 0.15, 'w2': 0.35, 'w3': 0.20, 'w4': 0.30},  # High avg, penalize injury
}

def compute_composite_score(data, weights):
    """Compute weighted composite score for each player."""
    score = (
        weights['w1'] * data['strike_rate'] +
        weights['w2'] * data['batting_avg'] +
        weights['w3'] * data['recent_form_score'] -
        weights['w4'] * data['injury_risk'] +
        data['role'].map(role_bonus)
    )
    return score

# Compute scores for all strategies
for name, weights in strategies.items():
    df[f'score_{name.lower()}'] = compute_composite_score(df_norm, weights).round(4)

print("=== Top 10 Players by Each Strategy ===\n")
for name in strategies:
    col = f'score_{name.lower()}'
    top10 = df.nlargest(10, col)[['player_name', 'role', 'strike_rate', 'batting_avg', 'base_price', col]]
    print(f"\n--- {name} Strategy ---")
    print(top10.to_string(index=False))

## Step 3: Sorting Algorithm Implementations

We implement three sorting algorithms and compare their behavior:
1. **Selection Sort** — O(n²): simple but slow; finds minimum each pass
2. **Insertion Sort** — O(n²) worst but O(n) best: efficient for nearly-sorted data
3. **QuickSort** — O(n log n) average: fast, in-place, industry standard

In [ ]:
def selection_sort(arr, key_index):
    """Selection Sort: O(n²) — find min/max in unsorted portion, swap to front."""
    data = [list(row) for row in arr]
    n = len(data)
    comparisons = 0
    for i in range(n):
        max_idx = i
        for j in range(i + 1, n):
            comparisons += 1
            if data[j][key_index] > data[max_idx][key_index]:
                max_idx = j
        data[i], data[max_idx] = data[max_idx], data[i]
    return data, comparisons

def insertion_sort(arr, key_index):
    """Insertion Sort: O(n²) worst, O(n) best — efficient for nearly-sorted data."""
    data = [list(row) for row in arr]
    n = len(data)
    comparisons = 0
    for i in range(1, n):
        key = data[i]
        j = i - 1
        while j >= 0:
            comparisons += 1
            if data[j][key_index] < key[key_index]:
                data[j + 1] = data[j]
                j -= 1
            else:
                break
        data[j + 1] = key
    return data, comparisons

def quicksort(arr, key_index):
    """QuickSort: O(n log n) average — divide and conquer with pivot."""
    comparisons = [0]  # mutable counter

    def _quicksort(data, low, high):
        if low < high:
            pivot_idx = partition(data, low, high)
            _quicksort(data, low, pivot_idx - 1)
            _quicksort(data, pivot_idx + 1, high)

    def partition(data, low, high):
        pivot = data[high][key_index]
        i = low - 1
        for j in range(low, high):
            comparisons[0] += 1
            if data[j][key_index] >= pivot:
                i += 1
                data[i], data[j] = data[j], data[i]
        data[i + 1], data[high] = data[high], data[i + 1]
        return i + 1

    data = [list(row) for row in arr]
    _quicksort(data, 0, len(data) - 1)
    return data, comparisons[0]

# Test with balanced strategy scores
score_col = 'score_balanced'
score_idx = list(df.columns).index(score_col)
arr_data = df.values.tolist()

# Run all three sorts
results = {}
for name, sort_fn in [('Selection Sort', selection_sort),
                       ('Insertion Sort', insertion_sort),
                       ('QuickSort', quicksort)]:
    start = time.perf_counter()
    sorted_data, comps = sort_fn(arr_data, score_idx)
    elapsed = time.perf_counter() - start
    results[name] = {'time': elapsed, 'comparisons': comps}
    print(f"{name:20s} | Time: {elapsed:.6f}s | Comparisons: {comps:,}")

# Verify: also time Python's built-in sort (TimSort)
start = time.perf_counter()
df_sorted = df.sort_values(score_col, ascending=False).reset_index(drop=True)
builtin_time = time.perf_counter() - start
print(f"{'Python TimSort':20s} | Time: {builtin_time:.6f}s | (built-in, optimized C)")

## Step 4: Scaling Simulation — Performance at Different Dataset Sizes

How do these algorithms behave as the dataset grows? We'll simulate sorting at 200, 500, 1K, 2K, and 5K records.

In [ ]:
# Simulate at different scales by replicating & adding noise
sizes = [200, 500, 1000, 2000, 5000]
timing_results = {alg: [] for alg in ['Selection Sort', 'Insertion Sort', 'QuickSort', 'Python TimSort']}

for size in sizes:
    # Create synthetic data at this scale
    if size <= len(df):
        test_data = df.head(size).values.tolist()
    else:
        # Replicate with noise
        reps = (size // len(df)) + 1
        extended = pd.concat([df] * reps, ignore_index=True).head(size)
        extended[score_col] += np.random.uniform(-0.01, 0.01, size)
        test_data = extended.values.tolist()

    for alg_name, sort_fn in [('Selection Sort', selection_sort),
                                ('Insertion Sort', insertion_sort),
                                ('QuickSort', quicksort)]:
        start = time.perf_counter()
        sort_fn(test_data, score_idx)
        elapsed = time.perf_counter() - start
        timing_results[alg_name].append(elapsed)

    # Python built-in
    test_df = pd.DataFrame(test_data, columns=df.columns)
    start = time.perf_counter()
    test_df.sort_values(score_col, ascending=False)
    elapsed = time.perf_counter() - start
    timing_results['Python TimSort'].append(elapsed)

    print(f"Size {size:5d}: Selection={timing_results['Selection Sort'][-1]:.4f}s | "
          f"Insertion={timing_results['Insertion Sort'][-1]:.4f}s | "
          f"Quick={timing_results['QuickSort'][-1]:.4f}s | "
          f"TimSort={timing_results['Python TimSort'][-1]:.6f}s")

In [ ]:
# Visualize: Execution Time vs Dataset Size
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Selection Sort': '#EF4444', 'Insertion Sort': '#F59E0B',
          'QuickSort': '#10B981', 'Python TimSort': '#2563EB'}

# Linear scale
ax1 = axes[0]
for alg, times in timing_results.items():
    ax1.plot(sizes, times, marker='o', label=alg, color=colors[alg], linewidth=2)
ax1.set_xlabel('Dataset Size (n)', fontsize=12)
ax1.set_ylabel('Time (seconds)', fontsize=12)
ax1.set_title('Sorting Performance: Linear Scale', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Log scale
ax2 = axes[1]
for alg, times in timing_results.items():
    ax2.plot(sizes, times, marker='o', label=alg, color=colors[alg], linewidth=2)
ax2.set_xlabel('Dataset Size (n)', fontsize=12)
ax2.set_ylabel('Time (seconds, log scale)', fontsize=12)
ax2.set_title('Sorting Performance: Log Scale', fontsize=14, fontweight='bold')
ax2.set_yscale('log')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Strategy Comparison — How Rankings Change with Weights

In [ ]:
# Compare top-10 rankings across strategies
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (strat_name, _) in enumerate(strategies.items()):
    col = f'score_{strat_name.lower()}'
    top10 = df.nlargest(10, col)

    ax = axes[idx]
    bars = ax.barh(range(10), top10[col].values, color=plt.cm.viridis(np.linspace(0.3, 0.9, 10)))
    ax.set_yticks(range(10))
    ax.set_yticklabels([f"{name} ({role})" for name, role in
                        zip(top10['player_name'], top10['role'])], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Composite Score')
    ax.set_title(f'Top 10 — {strat_name} Strategy', fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Show how player rank changes across strategies
print("\n=== Rank Position Changes Across Strategies ===")
rank_comparison = pd.DataFrame()
for strat_name in strategies:
    col = f'score_{strat_name.lower()}'
    df_temp = df.sort_values(col, ascending=False).reset_index(drop=True)
    df_temp[f'rank_{strat_name.lower()}'] = range(1, len(df_temp) + 1)
    if rank_comparison.empty:
        rank_comparison = df_temp[['player_name', f'rank_{strat_name.lower()}']]
    else:
        rank_comparison = rank_comparison.merge(
            df_temp[['player_name', f'rank_{strat_name.lower()}']], on='player_name')

# Show players with largest rank swings
rank_comparison['rank_range'] = rank_comparison[['rank_aggressive', 'rank_balanced', 'rank_conservative']].max(axis=1) - \
                                 rank_comparison[['rank_aggressive', 'rank_balanced', 'rank_conservative']].min(axis=1)
print("\nPlayers with LARGEST rank swings across strategies:")
print(rank_comparison.nlargest(10, 'rank_range').to_string(index=False))

## Step 6: Sensitivity Analysis — Which Weight Matters Most?

Vary each weight parameter ±10% and measure how much the top-10 ranking shifts.

In [ ]:
# Sensitivity analysis: perturb each weight ±10% and measure rank change
base_weights = strategies['Balanced'].copy()
base_scores = compute_composite_score(df_norm, base_weights)
base_ranking = base_scores.rank(ascending=False)

weight_names = {'w1': 'Strike Rate', 'w2': 'Batting Avg', 'w3': 'Recent Form', 'w4': 'Injury Risk'}
perturbations = np.arange(-0.10, 0.11, 0.02)

sensitivity = {}
for w_key, w_label in weight_names.items():
    avg_rank_changes = []
    for delta in perturbations:
        perturbed = base_weights.copy()
        perturbed[w_key] = base_weights[w_key] + delta
        new_scores = compute_composite_score(df_norm, perturbed)
        new_ranking = new_scores.rank(ascending=False)
        avg_change = (new_ranking - base_ranking).abs().mean()
        avg_rank_changes.append(avg_change)
    sensitivity[w_label] = avg_rank_changes

# Plot sensitivity
fig, ax = plt.subplots(figsize=(10, 6))
for label, changes in sensitivity.items():
    ax.plot(perturbations * 100, changes, marker='o', label=label, linewidth=2)

ax.set_xlabel('Weight Perturbation (%)', fontsize=12)
ax.set_ylabel('Average Rank Position Change', fontsize=12)
ax.set_title('Sensitivity Analysis: Which Weight Parameter Moves Rankings Most?',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("\n📊 Business Insight: The parameter with the steepest slope has the MOST impact on rankings.")
print("This tells team management which scouting metric deserves the most analytical rigor.")

## Step 7: Budget-Constrained Squad Selection

Select the optimal squad of 11 players within a **₹100 Crore** budget, maximizing total composite score — a knapsack-style optimization.

In [ ]:
# Budget-constrained squad selection (greedy approach on value-for-money ratio)
BUDGET = 10000  # ₹100 Crore = 10000 Lakhs (base_price is in Lakhs)
SQUAD_SIZE = 11

# Value-for-money: score / base_price
df['value_ratio'] = df['score_balanced'] / df['base_price']
df_sorted_value = df.sort_values('value_ratio', ascending=False).reset_index(drop=True)

# Greedy selection
selected = []
total_cost = 0
for _, player in df_sorted_value.iterrows():
    if len(selected) >= SQUAD_SIZE:
        break
    if total_cost + player['base_price'] <= BUDGET:
        selected.append(player)
        total_cost += player['base_price']

squad = pd.DataFrame(selected)
print(f"=== Optimal Squad (Greedy, Balanced Strategy) ===")
print(f"Total budget used: ₹{total_cost} Lakhs / ₹{BUDGET} Lakhs")
print(f"Players selected: {len(squad)}")
print(f"Total composite score: {squad['score_balanced'].sum():.4f}\n")
print(squad[['player_name', 'role', 'strike_rate', 'batting_avg', 'base_price',
             'score_balanced', 'value_ratio']].to_string(index=False))

# Visualize squad composition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Role distribution
role_counts = squad['role'].value_counts()
axes[0].pie(role_counts.values, labels=role_counts.index, autopct='%1.0f%%',
            colors=plt.cm.Set2.colors, startangle=90)
axes[0].set_title('Squad Role Composition', fontsize=13, fontweight='bold')

# Score vs Price scatter
ax = axes[1]
for role in df['role'].unique():
    mask = df['role'] == role
    ax.scatter(df.loc[mask, 'base_price'], df.loc[mask, 'score_balanced'],
               alpha=0.3, label=role, s=30)
# Highlight selected squad
ax.scatter(squad['base_price'], squad['score_balanced'],
           color='red', s=120, marker='*', zorder=5, label='Selected Squad', edgecolors='black')
ax.set_xlabel('Base Price (₹ Lakhs)', fontsize=12)
ax.set_ylabel('Composite Score (Balanced)', fontsize=12)
ax.set_title('Value for Money: Score vs Price', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8: Strategic Analysis & Recommendations

### Discussion Points:
1. **Sensitivity Report:** Which weight parameter has the most impact on top-10 ranking? What does this mean for scouting?
2. **Bidding Strategy:** Recommend a budget allocation under ₹100 Cr constraint — justify with data
3. **Risk Analysis:** What if your top-ranked player gets injured? Backup strategy?
4. **Cross-Functional Integration:** Connect analytics to Finance (ROI per player), Operations (squad balance), and Strategy (win probability)
5. **Analytics vs Expert:** Where do data-driven rankings diverge from expert opinion, and why?

In [ ]:
# === Strategic Recommendation Summary ===

print("=" * 70)
print("STRATEGIC RECOMMENDATION SUMMARY")
print("=" * 70)

# 1. Most sensitive weight
print("\n1. SENSITIVITY INSIGHT:")
max_sensitivity = max(sensitivity.items(), key=lambda x: max(x[1]))
print(f"   Most impactful weight: {max_sensitivity[0]}")
print(f"   → A ±10% change shifts average rank by {max(max_sensitivity[1]):.1f} positions")
print(f"   → Recommendation: Invest most scouting resources in accurately measuring {max_sensitivity[0]}")

# 2. Budget efficiency
print(f"\n2. BUDGET EFFICIENCY:")
print(f"   Budget utilized: ₹{total_cost} / ₹{BUDGET} Lakhs ({total_cost/BUDGET*100:.1f}%)")
print(f"   Squad composite score: {squad['score_balanced'].sum():.3f}")
print(f"   Avg score per player: {squad['score_balanced'].mean():.3f}")
print(f"   Cost per score unit: ₹{total_cost / squad['score_balanced'].sum():.0f} Lakhs")

# 3. Risk analysis - what if top player gets injured?
print(f"\n3. RISK ANALYSIS:")
top_player = squad.nlargest(1, 'score_balanced').iloc[0]
print(f"   Top player: {top_player['player_name']} (Score: {top_player['score_balanced']:.3f})")
# Find best replacement not in squad
remaining = df[~df['player_name'].isin(squad['player_name'])]
budget_remaining = BUDGET - total_cost + top_player['base_price']
replacements = remaining[remaining['base_price'] <= budget_remaining].nlargest(3, 'score_balanced')
print(f"   If injured, top 3 replacements within budget:")
for _, r in replacements.iterrows():
    score_loss = ((top_player['score_balanced'] - r['score_balanced']) / top_player['score_balanced'] * 100)
    print(f"   → {r['player_name']} ({r['role']}) | Score: {r['score_balanced']:.3f} | Loss: {score_loss:.1f}%")

# 4. Algorithm recommendation
print(f"\n4. ALGORITHM RECOMMENDATION:")
print(f"   For 200 players (auction day): Any algorithm works — QuickSort recommended for consistency")
print(f"   For 5000+ player database: QuickSort or TimSort mandatory — O(n²) adds unacceptable latency")
print(f"   Cloud cost impact: At 10K records, Selection Sort costs ~{timing_results['Selection Sort'][-1]/timing_results['QuickSort'][-1]:.0f}x more compute than QuickSort")